In [11]:
import csv
import numpy as np
import re

In [ ]:
# Load the 3.5 vs. 4o-mini results:
response_35 = {}
response_4o = {}
response_35_repr = {}
with open ('/home/davidchan/Projects/dementor/disguising/old_comparison_results.csv', 'r') as f:
    # Columns are prompt,gpt35_response,gpt4omini_response,comparison_results
    for row in csv.DictReader(f):
        response_35[row['prompt']] = row['gpt35_response']
        response_4o[row['prompt']] = row['gpt4omini_response']

with open('/home/davidchan/Projects/dementor/disguising/new_comparison_results.csv', 'r') as f:
    # Columns are prompt,gpt35_reprompted,gpt4omini_response,comparison_results
    for row in csv.DictReader(f):
        response_35_repr[row['prompt']] = row['gpt35_reprompted']
        response_4o[row['prompt']] = row['gpt4omini_response']


In [9]:
# Compute mean and std of the lengths of the responses
response_35_lengths = [len(response) for response in response_35.values()]
response_4o_lengths = [len(response) for response in response_4o.values()]
response_35_repr_lengths = [len(response) for response in response_35_repr.values()]
mean_35 = np.mean(response_35_lengths)
std_35 = np.std(response_35_lengths)
mean_4o = np.mean(response_4o_lengths)
std_4o = np.std(response_4o_lengths)
mean_35_repr = np.mean(response_35_repr_lengths)
std_35_repr = np.std(response_35_repr_lengths)
print(f'GPT-3.5 mean response length: {mean_35:.2f} ± {std_35:.2f}')
print(f'GPT-4o mean response length: {mean_4o:.2f} ± {std_4o:.2f}')
print(f'GPT-3.5 repr mean response length: {mean_35_repr:.2f} ± {std_35_repr:.2f}')

GPT-3.5 mean response length: 731.66 ± 703.88
GPT-4o mean response length: 1602.40 ± 1904.45
GPT-3.5 repr mean response length: 558.27 ± 649.75


In [ ]:
def has_markdown(text):
    markdown_patterns = [
        r'\*\*.*?\*\*',       # bold **
        r'\*.*?\*',           # italic *
        r'\_\_.*?\_\_',       # bold __
        r'\_.*?\_',           # italic _
        r'\#',                # headers #
        r'\[.*?\]\(.*?\)',    # links [text](url)
        r'\!\[.*?\]\(.*?\)',  # images ![alt](url)
        r'\`{1,3}.*?\`{1,3}', # inline or code block `
        r'\>\s',              # blockquotes >
        r'\-\s',              # lists -
        r'\+\s',              # lists +
        r'\*\s',              # lists *
    ]

    return any(re.search(pattern, text) for pattern in markdown_patterns)

# Check if the responses contain markdown
markdown_35 = [has_markdown(response) for response in response_35.values()]
markdown_4o = [has_markdown(response) for response in response_4o.values()]
markdown_35_repr = [has_markdown(response) for response in response_35_repr.values()]

# Calculate the percentage of responses containing markdown
markdown_35_percentage = (sum(markdown_35) / len(markdown_35)) * 100
markdown_4o_percentage = (sum(markdown_4o) / len(markdown_4o)) * 100
markdown_35_repr_percentage = (sum(markdown_35_repr) / len(markdown_35_repr)) * 100

print(f'Percentage of GPT-3.5 responses containing markdown: {markdown_35_percentage:.2f}%')
print(f'Percentage of GPT-4o responses containing markdown: {markdown_4o_percentage:.2f}%')
print(f'Percentage of GPT-3.5 repr responses containing markdown: {markdown_35_repr_percentage:.2f}%')

Percentage of GPT-3.5 responses containing markdown: 20.46%
Percentage of GPT-4o responses containing markdown: 61.35%
Percentage of GPT-3.5 repr responses containing markdown: 42.89%


In [ ]:

def contains_any_list(text, relaxed=False):
    strict_patterns = [
        r'(^|\n)[\-\*\+]\s+',           # Markdown unordered
        r'(^|\n)\d+\.\s+',              # Markdown ordered
        r'<(ul|ol|li)[^>]*>',           # HTML list tags
        r'(^|\n)\s*[•‣◦▪–—-]\s+',       # Unicode bullets
    ]

    relaxed_patterns = [
        r'(^|\n)[a-zA-Z][\.\)]\s+',                 # Alphabetical lists (a. or b))
        r'(^|\n)[IVXLCDMivxlcdm]{1,5}[\.\)]\s+',     # Roman numerals
        r'(^|\n)\s*[-]\s+\w+',                      # YAML-style lists
        r'\[\s*".+?"(?:,\s*".+?")+\s*\]',           # Simple JSON array
        r'(^|\n)\s*[→➡️🔹📌✅➤•]\s+',              # Emoji / symbol bullets
    ]

    patterns = strict_patterns + relaxed_patterns if relaxed else strict_patterns
    return any(re.search(pattern, text) for pattern in patterns)

# Check if the responses contain lists
list_35 = [contains_any_list(response) for response in response_35.values()]
list_4o = [contains_any_list(response) for response in response_4o.values()]
list_35_repr = [contains_any_list(response) for response in response_35_repr.values()]

# Calculate the percentage of responses containing lists
list_35_percentage = (sum(list_35) / len(list_35)) * 100
list_4o_percentage = (sum(list_4o) / len(list_4o)) * 100
list_35_repr_percentage = (sum(list_35_repr) / len(list_35_repr)) * 100

print(f'Percentage of GPT-3.5 responses containing lists: {list_35_percentage:.2f}%')
print(f'Percentage of GPT-4o responses containing lists: {list_4o_percentage:.2f}%')
print(f'Percentage of GPT-3.5 repr responses containing lists: {list_35_repr_percentage:.2f}%')

# Check if the responses contain lists (relaxed)
list_35_relaxed = [contains_any_list(response, relaxed=True) for response in response_35.values()]
list_4o_relaxed = [contains_any_list(response, relaxed=True) for response in response_4o.values()]
list_35_repr_relaxed = [contains_any_list(response, relaxed=True) for response in response_35_repr.values()]

# Calculate the percentage of responses containing lists (relaxed)
list_35_relaxed_percentage = (sum(list_35_relaxed) / len(list_35_relaxed)) * 100
list_4o_relaxed_percentage = (sum(list_4o_relaxed) / len(list_4o_relaxed)) * 100
list_35_repr_relaxed_percentage = (sum(list_35_repr_relaxed) / len(list_35_repr_relaxed)) * 100

print(f'Percentage of GPT-3.5 responses containing lists (relaxed): {list_35_relaxed_percentage:.2f}%')
print(f'Percentage of GPT-4o responses containing lists (relaxed): {list_4o_relaxed_percentage:.2f}%')
print(f'Percentage of GPT-3.5 repr responses containing lists (relaxed): {list_35_repr_relaxed_percentage:.2f}%')


Percentage of GPT-3.5 responses containing lists: 22.15%
Percentage of GPT-4o responses containing lists: 52.15%
Percentage of GPT-3.5 repr responses containing lists: 34.27%
Percentage of GPT-3.5 responses containing lists (relaxed): 22.65%
Percentage of GPT-4o responses containing lists (relaxed): 52.44%
Percentage of GPT-3.5 repr responses containing lists (relaxed): 34.68%


In [4]:

Lengths4o = []
Lengths35 = []
Lengths35_reprompt = []

with open('/home/davidchan/Projects/dementor/disguising/model_responses.csv', 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Columns: prompt,gpt4o_response,gpt35_response,gpt35_reprompted
        # Strip whitespace from the responses
        gpt4o_response = row['gpt4o_response'].strip()
        gpt35_response = row['gpt35_response'].strip()
        gpt35_reprompted_response = row['gpt35_reprompted'].strip()

        # Calculate the lengths of the responses
        length_4o = len(gpt4o_response.split())
        length_35 = len(gpt35_response.split())
        length_35_reprompted = len(gpt35_reprompted_response.split())

        # Append the lengths to the respective lists
        Lengths4o.append(length_4o)
        Lengths35.append(length_35)
        Lengths35_reprompt.append(length_35_reprompted)


# Calculate the average lengths
average_length_4o = sum(Lengths4o) / len(Lengths4o)
average_length_35 = sum(Lengths35) / len(Lengths35)
average_length_35_reprompt = sum(Lengths35_reprompt) / len(Lengths35_reprompt)

# Calculate the standard deviations
std_dev_4o = np.std(Lengths4o)
std_dev_35 = np.std(Lengths35)
std_dev_35_reprompt = np.std(Lengths35_reprompt)

# Print the results
print(f"Average length of GPT-4o responses: {average_length_4o:.2f} +/- {std_dev_4o:.2f} words")
print(f"Average length of GPT-3.5 responses: {average_length_35:.2f} +/- {std_dev_35:.2f} words")
print(f"Average length of GPT-3.5 (re-prompted) responses: {average_length_35_reprompt:.2f} +/- {std_dev_35_reprompt:.2f} words")


Average length of GPT-4o responses: 216.08 +/- 171.44 words
Average length of GPT-3.5 responses: 113.67 +/- 106.57 words
Average length of GPT-3.5 (re-prompted) responses: 85.26 +/- 93.94 words


In [7]:
# Count presence of markdown (i.e. number of "#" in the response)
def count_markdown(response):
    return response.count('#')

with open('/home/davidchan/Projects/dementor/disguising/model_responses.csv', 'r') as f:
    reader = csv.DictReader(f)
    markdown_counts_4o = [count_markdown(row['gpt4o_response'].strip()) for row in reader]
    markdown_counts_35 = [count_markdown(row['gpt35_response'].strip()) for row in reader]
    markdown_counts_35_reprompt = [count_markdown(row['gpt35_reprompted'].strip()) for row in reader]

    # Calculate the average markdown counts
    average_markdown_4o = sum(markdown_counts_4o) / len(markdown_counts_4o) if markdown_counts_4o else 0
    average_markdown_35 = sum(markdown_counts_35) / len(markdown_counts_35) if markdown_counts_35 else 0
    average_markdown_35_reprompt = sum(markdown_counts_35_reprompt) / len(markdown_counts_35_reprompt) if markdown_counts_35_reprompt else 0

    # Calculate the standard deviations
    std_dev_markdown_4o = np.std(markdown_counts_4o)
    std_dev_markdown_35 = np.std(markdown_counts_35)
    std_dev_markdown_35_reprompt = np.std(markdown_counts_35_reprompt)

    # Print the results
    print(f"Average markdown count in GPT-4o responses: {average_markdown_4o:.2f} +/- {std_dev_markdown_4o:.2f}")
    print(f"Average markdown count in GPT-3.5 responses: {average_markdown_35:.2f} +/- {std_dev_markdown_35:.2f}")
    print(f"Average markdown count in GPT-3.5 (re-prompted) responses: {average_markdown_35_reprompt:.2f} +/- {std_dev_markdown_35_reprompt:.2f}")

Average markdown count in GPT-4o responses: 1.74 +/- 5.65
Average markdown count in GPT-3.5 responses: 0.00 +/- nan
Average markdown count in GPT-3.5 (re-prompted) responses: 0.00 +/- nan


/home/davidchan/Apps/miniconda3/envs/dementor/lib/python3.12/site-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/davidchan/Apps/miniconda3/envs/dementor/lib/python3.12/site-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/davidchan/Apps/miniconda3/envs/dementor/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
